# AuthentiVision: MobileNetV2 Deep Learning & Explainable AI Training
### AI-Generated Content Detection in Digital Marketing

This Google Colab notebook provides the complete, self-contained training and evaluation pipeline for **AuthentiVision** using the balanced **CIFAKE (50K Train / 20K Test)** dataset.

#### Architecture Overview:
- **Base Backbone**: MobileNetV2 (Pretrained on ImageNet, backbone frozen)
- **Input Resolution**: 224 × 224 × 3
- **Pooling**: `GlobalAveragePooling2D`
- **Regularization**: `Dropout(0.3)`
- **Classification Head**: `Dense(1, activation='sigmoid')` (Probability of AI-Generated Content)
- **Loss Function**: `BinaryCrossentropy`
- **Optimizer**: `Adam(learning_rate=0.001)`
- **Target Deliverable**: `authentivision_mobilenetv2.h5`

In [ ]:
# 1. Environment Verification & GPU Check
!nvidia-smi

import os
import sys
import json
import numpy as np
import tensorflow as tf

print(f"TensorFlow Version: {tf.__version__}")
print(f"Available GPUs: {tf.config.list_physical_devices('GPU')}")

## 2. Dataset Extraction
Upload `CIFAKE_Train50K_Test20K.zip` to Colab or mount Google Drive.

In [ ]:
# Extract dataset archive
ZIP_PATH = "CIFAKE_Train50K_Test20K.zip"
DATA_DIR = "dataset"

if os.path.exists(ZIP_PATH):
    print(f"Extracting {ZIP_PATH}...")
    !unzip -q {ZIP_PATH} -d {DATA_DIR}
    print("Extraction complete!")
else:
    print(f"Please upload {ZIP_PATH} to the Colab working directory or mount Google Drive.")

## 3. Data Pipeline & Balance Verification
- **Training Set**: 40,000 images (20,000 REAL + 20,000 FAKE)
- **Validation Set**: 10,000 images (5,000 REAL + 5,000 FAKE)
- **Test Set**: 20,000 images (10,000 REAL + 10,000 FAKE)
- Class Mapping: `0 -> REAL`, `1 -> FAKE (AI-Generated)`

In [ ]:
BATCH_SIZE = 64
IMG_SIZE = (224, 224)
CLASS_NAMES = ["REAL", "FAKE"]

train_dir = os.path.join(DATA_DIR, "train")
test_dir = os.path.join(DATA_DIR, "test")

# 40K Training Split
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=42
)

# 10K Validation Split
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=42
)

# 20K Separate Test Split
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    class_names=CLASS_NAMES,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

# MobileNetV2 Normalization & Prefetching
preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess(x), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

print("Datasets successfully initialized and preprocessed!")

## 4. MobileNetV2 Transfer Learning Architecture

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Freeze the pretrained MobileNetV2 backbone
base_model.trainable = False

# Build classification head
x = base_model.output
x = tf.keras.layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
x = tf.keras.layers.Dropout(0.3, name="top_dropout")(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="predictions")(x)

model = tf.keras.Model(inputs=base_model.input, outputs=outputs, name="AuthentiVision_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Training with Callbacks & H5 Model Generation

In [ ]:
H5_MODEL_PATH = "authentivision_mobilenetv2.h5"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=H5_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=1,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

# Ensure model is saved to .h5
model.save(H5_MODEL_PATH)
print(f"Training complete. Model saved to: {H5_MODEL_PATH}")

## 6. Model Verification & Reload Testing

In [ ]:
# Load model using tf.keras.models.load_model with compile=False
reloaded_model = tf.keras.models.load_model(H5_MODEL_PATH, compile=False)
sample_input = tf.zeros((1, 224, 224, 3))
pred = reloaded_model.predict(sample_input, verbose=0)
print(f"Verification passed! Output shape: {pred.shape}, Sample output probability: {float(pred[0][0]):.4f}")

## 7. Comprehensive Evaluation on Untouched 20,000 Test Images

In [ ]:
y_true = []
y_pred_probs = []

for x_batch, y_batch in test_ds:
    p = reloaded_model.predict(x_batch, verbose=0)
    y_true.extend(y_batch.numpy().flatten())
    y_pred_probs.extend(p.flatten())

y_true = np.array(y_true, dtype=int)
y_probs = np.array(y_pred_probs, dtype=float)
y_pred = (y_probs >= 0.50).astype(int)

tp = int(np.sum((y_pred == 1) & (y_true == 1)))
tn = int(np.sum((y_pred == 0) & (y_true == 0)))
fp = int(np.sum((y_pred == 1) & (y_true == 0)))
fn = int(np.sum((y_pred == 0) & (y_true == 1)))

total = len(y_true)
total_real = int(np.sum(y_true == 0))
total_fake = int(np.sum(y_true == 1))

overall_acc = (tp + tn) / total
real_acc = tn / total_real
fake_acc = tp / total_fake
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 60)
print("CIFAKE 20,000 TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"Total Test Images:       {total:,}")
print(f"Overall Accuracy:        {overall_acc * 100:.2f}%")
print(f"REAL Accuracy:           {real_acc * 100:.2f}% ({tn:,}/{total_real:,})")
print(f"FAKE Accuracy:           {fake_acc * 100:.2f}% ({tp:,}/{total_fake:,})")
print(f"Precision (AI):          {precision * 100:.2f}%")
print(f"Recall (AI):             {recall * 100:.2f}%")
print(f"F1-Score (AI):           {f1 * 100:.2f}%")
print("-" * 60)
print("Confusion Matrix:")
print(f"  True REAL (TN):  {tn:,} | False FAKE (FP): {fp:,}")
print(f"  False REAL (FN): {fn:,} | True FAKE (TP):  {tp:,}")
print("=" * 60)

## 8. Download Trained .h5 Model
Run this cell to download `authentivision_mobilenetv2.h5` directly to your local computer and place it inside `models/`.

In [ ]:
from google.colab import files
files.download(H5_MODEL_PATH)